# MakeMeMeme - retrieval pipeline (Colab)

1. Mount Google Drive (images + catalog live there).
2. Clone the GitHub repo (all code).
3. Install deps (paddleocr 3.x + paddle GPU build for CUDA 12.x).
4. Configure paths via `%env`.
5. Download images -> OCR -> build index -> search / UI.

Runtime -> Change runtime type -> **GPU (T4)** before running.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Clone repo + install

Set `REPO_URL` to your fork. (Alternative: zip the repo (without `.venv`, `data`) and unzip it from Drive.)

Install order matters: `paddleocr` first (pulls the CPU paddle), then the **GPU** build overwrites it.

In [ ]:
%cd /content
REPO_URL = "https://github.com/abhijitdalal26/memefinder.git"  # <-- set your repo
import os
if not os.path.exists("/content/MakeMeMeme"):
    !git clone {REPO_URL}
%cd /content/MakeMeMeme
!pip install -q sentence-transformers gradio
!pip install -q paddleocr
!pip install -q paddlepaddle-gpu==3.3.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/

In [ ]:
import paddle
print("paddle", paddle.__version__, "| GPU compiled:", paddle.device.is_compiled_with_cuda())

## Configure paths (edit the Drive path to match yours)

In [ ]:
%env MAKEMEME_ROOT=/content/MakeMeMeme
%env MAKEMEME_IMAGES=/content/drive/MyDrive/MakeMeMeme/data
%env MAKEMEME_CATALOG=/content/drive/MyDrive/MakeMeMeme/curated_metadata.json
%env MAKEMEME_OCR=/content/drive/MyDrive/MakeMeMeme/ocr_cache.json
%env MAKEMEME_INDEX=/content/MakeMeMeme/search/index
%env MAKEMEME_SHARE=1

## 1. Download images (resumable)

In [ ]:
!python pipeline/download_images.py

## 2. OCR with PaddleOCR 3.x -> ocr_cache.json (resumable, GPU)

Batched (`MAKEMEME_OCR_BATCH`, default 16); skips ids already in the cache.
First run downloads PP-OCRv6 det/rec + textline-orientation models automatically.

In [ ]:
!python pipeline/ocr.py

## 3. Build the embedding index

In [ ]:
!python search/build_index.py

## 4. CLI smoke test

In [ ]:
!python -m search.cli "i don't want to work today"

## 5. Launch Gradio UI (public share link)

In [ ]:
!python -m search.app